# 06 · Build and maintain a RAG index

A vector database is not the source of truth. The source of truth is versioned documents; the index contains fragments, their embeddings, and pointers to files. In this lesson, we create a local Chroma, update one fragment, remove an outdated version, and verify a citation by hash. Each run uses a separate directory under `.tmp/rag`, and does not touch the Live Example data.

In [ ]:
# Only cell with imports and project path.
import hashlib
import json
import sys
import tempfile
from pathlib import Path

import chromadb
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, ollama_settings, section_records

## 1. Explicit index contract

The `document_id:version:section` is immutable for a specific version; a new version receives a new ID. The content hash detects changes, and the embedding model name identifies the vector space. Never mix vectors from different models in the same collection. `access_scope` is a didactic metadata; production authorization must be enforced by the application, not by the model.

In [ ]:
records = section_records()
host, embed_model = ollama_settings()
scratch = ROOT / ".tmp" / "rag"
scratch.mkdir(parents=True, exist_ok=True)
workspace = Path(tempfile.mkdtemp(prefix="index-", dir=scratch))
db = chromadb.PersistentClient(path=str(workspace / "chroma"))
collection = db.create_collection("acme-teaching-v1", configuration={"hnsw": {"space": "cosine"}},
                                  metadata={"embedding_model": embed_model})
len(records), workspace.name, embed_model

## 2. Ingest: text and metadata alongside the vector

The vector is generated from the section text. `upsert` is repeatable for the same ID, but if the content or the model changes, the embedding must be recalculated. Below, we build a manifest with hashes; the full system will compare it with the previous run and remove deprecated IDs.

In [ ]:
vectors = embed_texts([row["text"] for row in records])
collection.upsert(ids=[row["id"] for row in records], embeddings=vectors.tolist(),
                  documents=[row["text"] for row in records],
                  metadatas=[{key: row[key] for key in ("document_id", "version", "heading", "source_path", "access_scope", "content_hash")} for row in records])
manifest = {row["id"]: {"hash": row["content_hash"], "model": embed_model} for row in records}
(workspace / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
collection.count()

## 3. Update and delete without the old vector

We simulate a new version of *one section* in a separate `.tmp` file; we do not modify the actual document. The new ID points to version 2; the old ID is removed. First, prepare the file and the new vector, then perform the upsert, delete, and manifest update. These operations are not a single transaction: for large datasets, build a separate collection and switch the read source only after verifying the entire set.

In [ ]:
original = records[0]
revised = original["text"] + "\n\nTeaching revision: verify the reporting cutoff before comparing totals."
new_id = original["id"].replace(":v1:", ":v2:")
new_hash = hashlib.sha256(revised.encode("utf-8")).hexdigest()
revision_path = workspace / f"{original['document_id']}.v2.teaching-section.md"
revision_path.write_text(revised, encoding="utf-8")
new_metadata = {key: original[key] for key in ("document_id", "heading", "access_scope")}
new_metadata.update({"version": 2, "source_path": revision_path.relative_to(ROOT).as_posix(),
                     "content_hash": new_hash})
collection.upsert(ids=[new_id], embeddings=embed_texts([revised]).tolist(), documents=[revised], metadatas=[new_metadata])
collection.delete(ids=[original["id"]])
manifest.pop(original["id"])
manifest[new_id] = {"hash": new_hash, "model": embed_model}
(workspace / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
assert collection.get(ids=[original["id"]])["ids"] == []
assert collection.get(ids=[new_id])["ids"] == [new_id]
assert set(collection.get()["ids"]) == set(manifest)
collection.count(), revision_path.relative_to(ROOT).as_posix()

## 4. Retrieval and source verification

Chroma returns a candidate. After retrieving the record, we check the hash **and consistency with the source text**, as well as whether the ID is present in the current manifest. We separately fetch the new version by ID and compare the stored vector with a recalculated embedding. A correct update alone does not guarantee a high ranking for every query; the ANN result is checked separately. Version 2 is a local simulation, so it must not be cited as an actual Acme revision.

In [ ]:
query = embed_texts(["In the teaching revision, what should I verify before comparing totals?"])[0]
hits = collection.query(query_embeddings=[query.tolist()], n_results=3,
                        where={"access_scope": "finance"}, include=["documents", "metadatas", "distances"])
source_texts = {row["id"]: row["text"] for row in section_records()}
source_texts[new_id] = revision_path.read_text(encoding="utf-8")
verified = []
for identity, text, metadata, distance in zip(hits["ids"][0], hits["documents"][0], hits["metadatas"][0], hits["distances"][0]):
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == metadata["content_hash"]
    assert text == source_texts[identity]
    assert manifest[identity]["hash"] == metadata["content_hash"]
    assert (ROOT / metadata["source_path"]).is_file()
    verified.append({"id": identity, "source": metadata["source_path"], "distance": round(distance, 3)})
stored_revision = collection.get(ids=[new_id], include=["documents", "embeddings", "metadatas"])
assert stored_revision["documents"][0] == revision_path.read_text(encoding="utf-8")
revision_vector = embed_texts([revised])[0]
assert np.allclose(stored_revision["embeddings"][0], revision_vector, atol=1e-5)
probe = collection.query(query_embeddings=[revision_vector.tolist()], n_results=3,
                         where={"access_scope": "finance"}, include=["distances"])
{"question_top3": verified, "same_text_probe": probe["ids"][0]}

If `same_text_probe` does not place the new version in the first position, compare the Chroma result with the exact cosine similarity of the stored vectors and check the index status after reopening. `get(id)` confirms the record is saved; `query()` tests the search quality.

### Maintenance in Practice

Document change → detect hash → recalculate only changed fragments → remove old IDs → log the index version. Embedding model change → rebuild the **entire** index in a new collection. Add tests: number of records, orphaned IDs, hash consistency, access filters, recall on fixed queries. Monitor empty results and query drift. Do not store secrets in metadata or in the prompt.

**Exercise:** repeat the `upsert` and check if `count()` increases. Then remove the additional section and verify that it no longer appears in the search.